# 교육용 DNN 실습

이 노트북은 PyTorch를 사용한 세 가지 DNN 실습을 다룹니다.

| # | 태스크 | 데이터셋 | 손실함수 |
|---|--------|----------|---------|
| 1 | 회귀 | Diabetes | MSELoss |
| 2 | 다중분류 | Wine | CrossEntropyLoss |
| 3 | 희귀 이진분류 | Credit Card Fraud | BCEWithLogitsLoss / Focal Loss |

## 0. 환경 설정 및 공통 유틸리티

In [ ]:
# 필요 패키지 설치 (최초 1회)
# !pip install torch scikit-learn numpy

In [ ]:
import random
from typing import Dict, List, Tuple

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


def set_seed(seed: int = 42) -> None:
    """재현성을 위한 시드 고정"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def get_device() -> torch.device:
    """GPU 가능하면 cuda, 아니면 cpu"""
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


set_seed(42)
device = get_device()
print(f"Using device: {device}")

## 1. 공통 모델: MLP (Multi-Layer Perceptron)

### 핵심 이론
- **Linear → ReLU → Dropout** 블록을 반복하여 깊은 네트워크 구성
- **Dropout**: 훈련 시 뉴런을 무작위로 끄는 정규화 기법 → 과적합 방지
- **출력층**은 태스크에 따라 차원을 달리 설정
  - 회귀: `output_dim=1`
  - 다중분류: `output_dim=클래스 수`
  - 이진분류: `output_dim=1` (sigmoid 적용)

In [ ]:
class MLP(nn.Module):
    def __init__(
        self,
        input_dim: int,
        hidden_dims: List[int],
        output_dim: int,
        dropout: float = 0.1,
    ) -> None:
        super().__init__()
        layers: List[nn.Module] = []
        prev = input_dim
        for h in hidden_dims:
            layers.extend([nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)])
            prev = h
        layers.append(nn.Linear(prev, output_dim))  # 출력층 (활성화 없음)
        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


# 구조 확인 예시
sample_model = MLP(input_dim=10, hidden_dims=[64, 32], output_dim=1)
print(sample_model)

## 2. 공통 학습 루프

### 핵심 이론
- `optimizer=None`이면 **평가 모드** (가중치 갱신 없음)
- `torch.no_grad()`로 기울기 계산을 막아 메모리와 속도를 절약
- `optimizer.zero_grad()` → `loss.backward()` → `optimizer.step()` 순서가 중요

In [ ]:
def run_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer | None,
    device: torch.device,
) -> float:
    """범용 1-에폭 학습/평가 루프"""
    is_train = optimizer is not None
    model.train(is_train)  # train=True면 Dropout 활성화
    losses: List[float] = []

    context = torch.enable_grad() if is_train else torch.no_grad()
    with context:
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            pred = model(xb)
            loss = criterion(pred, yb)
            losses.append(float(loss.item()))

            if is_train:
                optimizer.zero_grad(set_to_none=True)  # 이전 기울기 초기화
                loss.backward()                         # 역전파
                optimizer.step()                        # 가중치 갱신

    return float(np.mean(losses))


print("run_epoch 함수 정의 완료")

---
## 실습 1: 회귀 — Diabetes 데이터셋

### 핵심 이론
- **손실함수**: MSELoss (평균 제곱 오차) → 연속값 예측에 사용
- **평가지표**: RMSE (루트 평균 제곱 오차) — 예측 오차의 단위가 타깃과 동일
- **데이터 분할**: 훈련 70% / 검증 15% / 테스트 15%
- **StandardScaler**: 평균 0, 표준편차 1로 정규화 (fit은 훈련셋에만!)

In [ ]:
from sklearn.datasets import load_diabetes
from sklearn.metrics import mean_squared_error


def make_regression_loaders(batch_size: int = 128):
    X, y = load_diabetes(return_X_y=True)

    # 70 / 15 / 15 분할
    X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42)
    X_valid, X_test, y_valid, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42)

    # 정규화 (fit은 훈련셋에만 적용)
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_valid = scaler.transform(X_valid)
    X_test  = scaler.transform(X_test)

    def to_loader(X, y, shuffle):
        xt = torch.tensor(X, dtype=torch.float32)
        yt = torch.tensor(y, dtype=torch.float32).unsqueeze(1)  # (N,) → (N,1)
        return DataLoader(TensorDataset(xt, yt), batch_size=batch_size, shuffle=shuffle)

    return (
        to_loader(X_train, y_train, shuffle=True),
        to_loader(X_valid, y_valid, shuffle=False),
        to_loader(X_test,  y_test,  shuffle=False),
        X_train.shape[1],  # input_dim
    )


def evaluate_regression(model, loader, device) -> Dict[str, float]:
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for xb, yb in loader:
            pred = model(xb.to(device)).cpu().numpy().squeeze()
            y_pred.append(np.atleast_1d(pred))
            y_true.append(yb.numpy().squeeze())
    y_true = np.concatenate(y_true)
    y_pred = np.concatenate(y_pred)
    return {"RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred)))}


print("회귀 데이터 로더 함수 정의 완료")

In [ ]:
# ── 회귀 학습 실행 ──────────────────────────────────────────
EPOCHS = 40
LR     = 1e-3

train_loader, valid_loader, test_loader, input_dim = make_regression_loaders()

reg_model = MLP(input_dim=input_dim, hidden_dims=[64, 32], output_dim=1, dropout=0.1).to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.AdamW(reg_model.parameters(), lr=LR, weight_decay=1e-4)

for epoch in range(1, EPOCHS + 1):
    train_loss = run_epoch(reg_model, train_loader, criterion, optimizer, device)
    valid_loss = run_epoch(reg_model, valid_loader, criterion, None, device)
    if epoch == 1 or epoch % 10 == 0:
        print(f"Epoch {epoch:03d} | train_loss={train_loss:.4f} | valid_loss={valid_loss:.4f}")

metrics = evaluate_regression(reg_model, test_loader, device)
print(f"\n[테스트 결과] RMSE = {metrics['RMSE']:.4f}")

---
## 실습 2: 다중분류 — Wine 데이터셋

### 핵심 이론
- **손실함수**: CrossEntropyLoss = LogSoftmax + NLLLoss (내부적으로 계산)
  - 입력: 로짓(logit) 벡터 `(N, C)`, 타깃: 정수 레이블 `(N,)`
- **stratify**: 클래스 비율을 유지하며 분할 → 소수 클래스 누락 방지
- **평가지표**: Accuracy, Macro-F1 (클래스 불균형 시 F1이 더 신뢰적)

In [ ]:
from sklearn.datasets import load_wine
from sklearn.metrics import accuracy_score, f1_score


def make_wine_loaders(batch_size: int = 128):
    X, y = load_wine(return_X_y=True)

    # stratify로 클래스 비율 보존
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=0.30, random_state=42, stratify=y
    )
    X_valid, X_test, y_valid, y_test = train_test_split(
        X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
    )

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_valid = scaler.transform(X_valid)
    X_test  = scaler.transform(X_test)

    def to_loader(X, y, shuffle):
        xt = torch.tensor(X, dtype=torch.float32)
        yt = torch.tensor(y, dtype=torch.long)  # CrossEntropyLoss는 long 타입 필요
        return DataLoader(TensorDataset(xt, yt), batch_size=batch_size, shuffle=shuffle)

    return (
        to_loader(X_train, y_train, shuffle=True),
        to_loader(X_valid, y_valid, shuffle=False),
        to_loader(X_test,  y_test,  shuffle=False),
        X_train.shape[1],
    )


def evaluate_multiclass(model, loader, device) -> Dict[str, float]:
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for xb, yb in loader:
            logits = model(xb.to(device))
            pred = logits.argmax(dim=1).cpu().numpy()  # 가장 높은 로짓 → 예측 클래스
            y_pred.append(pred)
            y_true.append(yb.numpy())
    y_true = np.concatenate(y_true)
    y_pred = np.concatenate(y_pred)
    return {
        "Accuracy": float(accuracy_score(y_true, y_pred)),
        "Macro-F1": float(f1_score(y_true, y_pred, average="macro")),
    }


print("다중분류 데이터 로더 함수 정의 완료")

In [ ]:
# ── 다중분류 학습 실행 ─────────────────────────────────────
train_loader, valid_loader, test_loader, input_dim = make_wine_loaders()

wine_model = MLP(input_dim=input_dim, hidden_dims=[64, 32], output_dim=3, dropout=0.1).to(device)
criterion  = nn.CrossEntropyLoss()
optimizer  = torch.optim.AdamW(wine_model.parameters(), lr=LR, weight_decay=1e-4)

for epoch in range(1, EPOCHS + 1):
    train_loss = run_epoch(wine_model, train_loader, criterion, optimizer, device)
    valid_loss = run_epoch(wine_model, valid_loader, criterion, None, device)
    if epoch == 1 or epoch % 10 == 0:
        print(f"Epoch {epoch:03d} | train_loss={train_loss:.4f} | valid_loss={valid_loss:.4f}")

metrics = evaluate_multiclass(wine_model, test_loader, device)
print(f"\n[테스트 결과]")
for k, v in metrics.items():
    print(f"  {k}: {v:.4f}")

---
## 실습 3: 희귀 이진분류 — Credit Card Fraud

### 핵심 이론: 클래스 불균형 대응 전략

신용카드 사기 데이터는 정상:사기 비율이 약 **500:1**로 극도로 불균형합니다.

#### 전략 1 — WeightedRandomSampler (샘플링)
- 소수 클래스(사기)를 더 자주 샘플링해 미니배치 내 클래스 비율 균형화
- `weight = 1 / class_count` 로 각 샘플의 가중치 계산

#### 전략 2 — BCEWithLogitsLoss + pos_weight
- 양성 샘플의 손실에 `pos_weight = neg/pos` 배 가중치 부여
- 모델이 소수 클래스를 무시하지 않도록 강제

#### 전략 3 — Focal Loss (선택)
- **쉬운 샘플**의 손실을 자동으로 낮추고 **어려운 샘플**에 집중
- `FL(pt) = α(1-pt)^γ · BCE(pt)`, γ=2 권장

#### 평가지표
- **ROC-AUC**: 임계값에 독립적인 순위 기반 지표
- **PR-AUC (Average Precision)**: 불균형 데이터에서 더 엄격한 지표
- **Recall**: 사기를 놓치지 않는 것이 목적이므로 핵심 지표

In [ ]:
class BinaryFocalLoss(nn.Module):
    """
    Binary Focal Loss (logit 입력 버전)
    FL(pt) = alpha_t * (1 - pt)^gamma * BCE(pt)
    """

    def __init__(self, alpha: float = 0.25, gamma: float = 2.0) -> None:
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        bce    = nn.functional.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        probs  = torch.sigmoid(logits)
        pt     = probs * targets + (1.0 - probs) * (1.0 - targets)      # 정답 클래스 확률
        alpha_t = self.alpha * targets + (1.0 - self.alpha) * (1.0 - targets)
        loss   = alpha_t * (1.0 - pt) ** self.gamma * bce
        return loss.mean()


print("Focal Loss 정의 완료")

In [ ]:
from sklearn.datasets import fetch_openml
from sklearn.metrics import roc_auc_score, average_precision_score, recall_score, f1_score


def make_fraud_loaders(batch_size: int = 128, max_samples: int | None = 30000):
    print("데이터 다운로드 중... (최초 실행 시 시간이 걸릴 수 있습니다)")
    bundle = fetch_openml(data_id=1597, as_frame=True)
    X = bundle.data.to_numpy(dtype=np.float32)
    y = bundle.target.astype(int).to_numpy()

    if max_samples and max_samples < len(X):
        X, _, y, _ = train_test_split(X, y, train_size=max_samples, stratify=y, random_state=42)

    X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, stratify=y, random_state=42)
    X_valid, X_test, y_valid, y_test = train_test_split(X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42)

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_valid = scaler.transform(X_valid)
    X_test  = scaler.transform(X_test)

    # WeightedRandomSampler: 소수 클래스 오버샘플링
    class_count    = np.bincount(y_train)
    sample_weights = (1.0 / np.maximum(class_count, 1))[y_train]
    sampler = WeightedRandomSampler(
        weights=torch.tensor(sample_weights, dtype=torch.float64),
        num_samples=len(sample_weights),
        replacement=True,
    )

    def to_tensor(X, y):
        return (torch.tensor(X, dtype=torch.float32),
                torch.tensor(y, dtype=torch.float32).unsqueeze(1))

    xt, yt = to_tensor(X_train, y_train)
    train_loader = DataLoader(TensorDataset(xt, yt), batch_size=batch_size, sampler=sampler)

    xv, yv = to_tensor(X_valid, y_valid)
    valid_loader = DataLoader(TensorDataset(xv, yv), batch_size=batch_size, shuffle=False)

    xe, ye = to_tensor(X_test, y_test)
    test_loader = DataLoader(TensorDataset(xe, ye), batch_size=batch_size, shuffle=False)

    return train_loader, valid_loader, test_loader, X_train.shape[1], y_train


def evaluate_binary(model, loader, device) -> Dict[str, float]:
    model.eval()
    y_true, y_prob = [], []
    with torch.no_grad():
        for xb, yb in loader:
            prob = torch.sigmoid(model(xb.to(device))).cpu().numpy().squeeze()
            y_prob.append(np.atleast_1d(prob))
            y_true.append(yb.numpy().squeeze())
    y_true = np.concatenate(y_true).astype(int)
    y_prob = np.concatenate(y_prob)
    y_pred = (y_prob >= 0.5).astype(int)
    return {
        "ROC-AUC": float(roc_auc_score(y_true, y_prob)),
        "PR-AUC":  float(average_precision_score(y_true, y_prob)),
        "Recall":  float(recall_score(y_true, y_pred, zero_division=0)),
        "F1":      float(f1_score(y_true, y_pred, zero_division=0)),
    }


print("희귀분류 함수 정의 완료")

In [ ]:
# ── 희귀 이진분류 학습 실행 ────────────────────────────────
# USE_FOCAL = True  → Focal Loss
# USE_FOCAL = False → BCEWithLogitsLoss + pos_weight
USE_FOCAL   = False
MAX_SAMPLES = 30000  # 전체 데이터 사용 시 None

train_loader, valid_loader, test_loader, input_dim, y_train = make_fraud_loaders(
    max_samples=MAX_SAMPLES
)

pos = float((y_train == 1).sum())
neg = float((y_train == 0).sum())
print(f"클래스 분포 → 정상: {int(neg)}, 사기: {int(pos)}, 비율: {neg/pos:.1f}:1")

fraud_model = MLP(input_dim=input_dim, hidden_dims=[128, 64], output_dim=1, dropout=0.15).to(device)
optimizer   = torch.optim.AdamW(fraud_model.parameters(), lr=LR, weight_decay=1e-4)

if USE_FOCAL:
    criterion = BinaryFocalLoss(alpha=0.75, gamma=2.0)
    print("손실함수: Focal Loss (alpha=0.75, gamma=2.0)")
else:
    pos_weight = torch.tensor([neg / max(pos, 1.0)], dtype=torch.float32, device=device)
    criterion  = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    print(f"손실함수: BCEWithLogitsLoss (pos_weight={float(pos_weight):.2f})")

for epoch in range(1, EPOCHS + 1):
    train_loss = run_epoch(fraud_model, train_loader, criterion, optimizer, device)
    valid_loss = run_epoch(fraud_model, valid_loader, criterion, None, device)
    if epoch == 1 or epoch % 10 == 0:
        print(f"Epoch {epoch:03d} | train_loss={train_loss:.4f} | valid_loss={valid_loss:.4f}")

metrics = evaluate_binary(fraud_model, test_loader, device)
print(f"\n[테스트 결과]")
for k, v in metrics.items():
    print(f"  {k:>8}: {v:.4f}")

---
## 정리: 태스크별 설계 비교

| 항목 | 회귀 | 다중분류 | 희귀 이진분류 |
|------|------|----------|--------------|
| 출력 노드 수 | 1 | 클래스 수 (3) | 1 |
| 출력 활성화 | 없음 (linear) | 없음 (CrossEntropy 내부 처리) | Sigmoid (평가 시) |
| 손실함수 | MSELoss | CrossEntropyLoss | BCEWithLogitsLoss / Focal Loss |
| 타깃 dtype | float32 + unsqueeze | long | float32 + unsqueeze |
| 불균형 대응 | - | stratify 분할 | Sampler + pos_weight / Focal Loss |
| 주요 지표 | RMSE | Accuracy, Macro-F1 | ROC-AUC, PR-AUC, Recall |